# Model Selection and Hyperparameter Tuning

**Use scikit-learn as much as possible!**

In [ ]:
import pandas as pd
import numpy as np

## Ames Housing Data

As usual, we'll try to predict SalePrice using a mix of quantitative and categorical features. See the [data documentation](https://jse.amstat.org/v19n3/decock/DataDocumentation.txt) for all the variables that are available in the data set. We'll try to come up with a model that minimizes test error, measured by root mean square error (RMSE).

In [ ]:
df_ames = pd.read_csv("https://raw.githubusercontent.com/kevindavisross/data301/main/data/AmesHousing.txt", sep="\t")
df_ames

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2925,2926,923275080,80,RL,37.0,7937,Pave,NaN,IR1,Lvl,...,0,NaN,GdPrv,NaN,0,3,2006,WD,Normal,142500
2926,2927,923276100,20,RL,NaN,8885,Pave,NaN,IR1,Low,...,0,NaN,MnPrv,NaN,0,6,2006,WD,Normal,131000
2927,2928,923400125,85,RL,62.0,10441,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,Shed,700,7,2006,WD,Normal,132000
2928,2929,924100070,20,RL,77.0,10010,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2006,WD,Normal,170000


1\. Recall that in a previous notebook we fit a 10-nearest neighbors regression model that predicts the price (just **SalePrice**, not log) of a home using square footage (**Gr Liv Area**), number of bedrooms (**Bedroom AbvGr**), number of full bathrooms (**Full Bath**), number of half bathrooms (**Half Bath**), and **Neighborhood**. But what about other sets of features?

Specify 5 sets of features to investigate. These sets could be subsets of the variables above, but you should also include other variables from the data set. The choice is up to you, but try to guess which sets might lead to good models based on your intuition and what we have already seen for this data set.

For each set of features, fit a 10-nearest neighbors regression model and estimate its test error (RMSE) using cross-validation. You can write a loop to iterate over the sets of features similar to what was done in the Model Selection section of the pre-class notebook.

Report your lowest RMSE and the corresponding features.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.model_selection import GridSearchCV, train_test_split

x = df_ames[["Gr Liv Area", "Bedroom AbvGr", "Full Bath", "Half Bath", "Neighborhood"]]
y = df_ames["SalePrice"]

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42)

ct = ColumnTransformer(
    [
        ("dummify",
         OneHotEncoder(sparse_output=False, handle_unknown='ignore'),
         make_column_selector(dtype_include="object")),
        ("standardize",
         StandardScaler(),
         make_column_selector(dtype_include=np.number))
    ],
    remainder="passthrough"
)

knn_pipeline = Pipeline(
    [("preprocessing", ct),
     ("knn", KNeighborsRegressor(n_neighbors=10))]
)


grid_search = GridSearchCV(
    knn_pipeline,
    param_grid={
        "knn__n_neighbors": range(1, 20)
    },
    scoring="neg_mean_squared_error",
    cv=10
)

grid_search.fit(x_train, y_train)
grid_search.best_estimator_

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('dummify',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7c405f358ad0>),
                                                 ('standardize',
                                                  StandardScaler(),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7c405f3586b0>)])),
                ('knn', KNeighborsRegressor())])

In [ ]:
df_cv_results_ = pd.DataFrame(grid_search.cv_results_)

df_cv_results_

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_knn__n_neighbors,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,split5_test_score,split6_test_score,split7_test_score,split8_test_score,split9_test_score,mean_test_score,std_test_score,rank_test_score
0,0.009198,0.000539,0.009313,0.000686,1,{'knn__n_neighbors': 1},-1.809328e+09,-2.101597e+09,-1.939209e+09,-1.902640e+09,-2.492718e+09,-1.609606e+09,-2.836691e+09,-1.641996e+09,-2.047439e+09,-1.759419e+09,-2.014064e+09,3.662514e+08,19
1,0.009357,0.000747,0.008985,0.000137,2,{'knn__n_neighbors': 2},-1.544823e+09,-2.113534e+09,-1.339370e+09,-1.396977e+09,-1.851972e+09,-1.458410e+09,-2.348495e+09,-1.514701e+09,-1.931031e+09,-1.238916e+09,-1.673823e+09,3.481998e+08,18
2,0.009698,0.001390,0.009765,0.001835,3,{'knn__n_neighbors': 3},-1.508423e+09,-1.872848e+09,-1.189716e+09,-1.402244e+09,-1.900235e+09,-1.086392e+09,-2.570747e+09,-1.379038e+09,-1.790345e+09,-1.035166e+09,-1.573515e+09,4.453802e+08,7
3,0.008895,0.000144,0.009066,0.000076,4,{'knn__n_neighbors': 4},-1.631993e+09,-1.931819e+09,-1.154312e+09,-1.323427e+09,-1.736367e+09,-1.034986e+09,-2.341245e+09,-1.300094e+09,-1.808035e+09,-1.090748e+09,-1.535303e+09,4.030186e+08,3
4,0.008864,0.000115,0.009288,0.000187,5,{'knn__n_neighbors': 5},-1.570940e+09,-2.010859e+09,-1.118862e+09,-1.324188e+09,-1.604267e+09,-1.073599e+09,-2.030931e+09,-1.295318e+09,-1.786600e+09,-1.121719e+09,-1.493728e+09,3.440203e+08,1
5,0.009230,0.000395,0.009700,0.000773,6,{'knn__n_neighbors': 6},-1.592844e+09,-1.986081e+09,-1.118788e+09,-1.398981e+09,-1.542588e+09,-1.099810e+09,-2.001147e+09,-1.375637e+09,-1.798615e+09,-1.225783e+09,-1.514027e+09,3.148652e+08,2
6,0.008976,0.000091,0.009256,0.000103,7,{'knn__n_neighbors': 7},-1.608376e+09,-1.977462e+09,-1.103421e+09,-1.502839e+09,-1.430858e+09,-1.134439e+09,-1.983985e+09,-1.442388e+09,-1.887854e+09,-1.284741e+09,-1.535636e+09,3.093609e+08,4
7,0.009402,0.001437,0.010652,0.002231,8,{'knn__n_neighbors': 8},-1.671892e+09,-1.983656e+09,-1.117569e+09,-1.567682e+09,-1.447768e+09,-1.212634e+09,-1.943414e+09,-1.558400e+09,-1.837275e+09,-1.336601e+09,-1.567689e+09,2.818286e+08,5
8,0.009035,0.000264,0.009531,0.000101,9,{'knn__n_neighbors': 9},-1.727652e+09,-1.984981e+09,-1.111182e+09,-1.609981e+09,-1.447826e+09,-1.236406e+09,-1.865505e+09,-1.549186e+09,-1.794139e+09,-1.394625e+09,-1.572148e+09,2.653473e+08,6
9,0.009179,0.000287,0.009749,0.000218,10,{'knn__n_neighbors': 10},-1.749298e+09,-2.047185e+09,-1.094080e+09,-1.687608e+09,-1.435184e+09,-1.279230e+09,-1.858406e+09,-1.572766e+09,-1.817300e+09,-1.400452e+09,-1.594151e+09,2.775184e+08,9


2\. Recall that in a previous notebook we fit a $k$-nearest neighbors regression model that predicts the price (just **SalePrice**, not log) of a home using square footage (**Gr Liv Area**), number of bedrooms (**Bedroom AbvGr**), number of full bathrooms (**Full Bath**), number of half bathrooms (**Half Bath**), and **Neighborhood**. Previously we used a loop to iterate and find the value of $k$ with the smallest test error (RMSE).

Now use grid search in scikit-learn to implement this process. Use the results to create a plot of test error versus $k$.

In [ ]:
knn_pipeline = Pipeline(
    [("preprocessing", ct),
     ("knn", KNeighborsRegressor())]
)

knn_grid_search = GridSearchCV(
    knn_pipeline,
    param_grid={
        "knn__n_neighbors": range(1, 20)
    },
    scoring="neg_mean_squared_error",
    cv=10
)

3\. $k$-nearest neighbors relies on distance and we know there are different distance metrics. Repeat the previous part and use grid search to iterate over both (1) `n_neighbors` and (2) `metric`, which can be `"euclidean"` or `"manhattan"` (see the the [`KNeighborsRegressor` documentation](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html)). Compare the results to the previous part. (Hint: recall the note in the pre-class notebook about grid search conventions for refering to steps in the pipeline.)

In [ ]:
knn_regression_pipeline = Pipeline(
    [("preprocessing", ct),
     ("knn", KNeighborsRegressor())]
)

Knn_grid_search = GridSearchCV(
    knn_regression_pipeline,
    param_grid={
        "knn__n_neighbors": range(1, 20),
        "knn__metric": ["euclidean", "manhattan"]
    }

    , scoring="neg_mean_squared_error",
    cv=10
)

Knn_grid_search.fit(x_train, y_train)
Knn_grid_search.best_estimator_

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('dummify',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7c405d0ab230>),
                                                 ('standardize',
                                                  StandardScaler(),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7c405d0ab290>)])),
                ('knn', KNeighborsRegressor(metric='manhattan'))])

4\. Combine elements of the previous parts and fit what you think will be the best $k$-nearest neighbors model. Specify which features you are using, the value of $k$, and the distance metric. Estimate the test error. (In 2-3 you used the specified set of features. Now you can combine elements of 2-3 with the features you chose in part 1, or other sets of features, as well as investigating other parameter choices.)

Note: Look at the [`KNeighborsRegressor` documentation](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html) for a full list of parameters. You might consider tuning `n_neighbors`, `metric` (or `p`), and/or `weights`. You could also try other elements like scaling too.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.model_selection import GridSearchCV, train_test_split


df_ames = pd.read_csv("https://raw.githubusercontent.com/kevindavisross/data301/main/data/AmesHousing.txt", sep="\t")

features = ["Gr Liv Area", "Bedroom AbvGr", "Full Bath", "Half Bath", "Neighborhood", "Overall Qual", "Year Built"]
x = df_ames[features]
y = df_ames["SalePrice"]

x = x.fillna(x.median(numeric_only=True))

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42)

ct = ColumnTransformer(
    [
        ("dummify",
         OneHotEncoder(sparse_output=False, handle_unknown='ignore'),
         make_column_selector(dtype_include="object")),
        ("standardize",
         StandardScaler(),
         make_column_selector(dtype_include=np.number))
    ],
    remainder="passthrough"
)

knn_pipeline = Pipeline([
    ("preprocessing", ct),
    ("knn", KNeighborsRegressor())
])

param_grid = {
    "knn__n_neighbors": range(1, 25),
    "knn__metric": ["euclidean", "manhattan"],
    "knn__weights": ["uniform", "distance"]
}

grid_search = GridSearchCV(
    knn_pipeline,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=10,
    n_jobs=-1
)

grid_search.fit(x_train, y_train)

best_rmse = np.sqrt(-grid_search.best_score_)
best_params = grid_search.best_params_

print(f"Selected Features: {features}")
print(f"Best Parameters: {best_params}")
print(f"Estimated Test RMSE (CV): ${best_rmse:.2f}")

Selected Features: ['Gr Liv Area', 'Bedroom AbvGr', 'Full Bath', 'Half Bath', 'Neighborhood', 'Overall Qual', 'Year Built']
Best Parameters: {'knn__metric': 'manhattan', 'knn__n_neighbors': 7, 'knn__weights': 'distance'}
Estimated Test RMSE (CV): $31216.03


5\. Now fit a linear regression model using the same features that you used in the previous part. Estimate the test error. How does the test error for linear regression compare to the kNN model?


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

lr_pipeline = Pipeline([
    ("preprocessing", ct),
    ("lr", LinearRegression())
])

lr_scores = cross_val_score(
    lr_pipeline,
    x_train,
    y_train,
    scoring="neg_mean_squared_error",
    cv=10,
    n_jobs=-1
)

lr_rmse = np.sqrt(-lr_scores.mean())

print(f"Linear Regression CV RMSE: ${lr_rmse:.2f}")
print(f"Best kNN CV RMSE: ${best_rmse:.2f}")
print(f"Difference: ${abs(best_rmse - lr_rmse):.2f} in favor of {'kNN' if best_rmse < lr_rmse else 'Linear Regression'}")

Linear Regression CV RMSE: $34628.80
Best kNN CV RMSE: $31216.03
Difference: $3412.77 in favor of kNN


## Loans Data

The loans data set represents thousands of loans made through the Lending Club platform, which is a platform that allows individuals to lend to other individuals. Of course, not all loans are created equal. Someone who is a essentially a sure bet to pay back a loan will have an easier time getting a loan with a low interest rate than someone who appears to be riskier. And for people who are very risky? They may not even get a loan offer, or they may not have accepted the loan offer due to a high interest rate. It is important to keep that last part in mind, since this dataset only represents loans actually made, i.e. do not mistake this data for loan applications!

[Some information about the data set is here](https://openintrostat.github.io/openintro/reference/loans_full_schema.html). There are also some related exercises in this [OpenIntro textbook](https://openintro-ims.netlify.app/inf-model-mlr.html).

**Our goal will be to develop a regression model for predicting `interest_rate`.**

We will have a competition to see who can come up with the model with the smallest test RMSE. Fit several models and try to find the smallest test RMSE. When you think you have a good one, share your model and its test RMSE. Then see what other people have done; can you do even better?

Note: when you are consider features to include, you might want to consider transformations of the variables.

You should use `GridSearchCV` to tune at least one parameter.

Note: you should do some exploration of the data first before you start fitting models.

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/kevindavisross/data301/refs/heads/main/data/loans_full_schema.csv")

df

,Unnamed: 0,emp_title,emp_length,state,homeownership,annual_income,verified_income,debt_to_income,annual_income_joint,verification_income_joint,...,sub_grade,issue_month,loan_status,initial_listing_status,disbursement_method,balance,paid_total,paid_principal,paid_interest,paid_late_fees
0,1,global config engineer,3.0,NJ,MORTGAGE,90000.0,Verified,18.01,NaN,NaN,...,C3,Mar-2018,Current,whole,Cash,27015.86,1999.33,984.14,1015.19,0.0
1,2,warehouse office clerk,10.0,HI,RENT,40000.0,Not Verified,5.04,NaN,NaN,...,C1,Feb-2018,Current,whole,Cash,4651.37,499.12,348.63,150.49,0.0
2,3,assembly,3.0,WI,RENT,40000.0,Source Verified,21.15,NaN,NaN,...,D1,Feb-2018,Current,fractional,Cash,1824.63,281.80,175.37,106.43,0.0
3,4,customer service,1.0,PA,RENT,30000.0,Not Verified,10.16,NaN,NaN,...,A3,Jan-2018,Current,whole,Cash,18853.26,3312.89,2746.74,566.15,0.0
4,5,security supervisor,10.0,CA,RENT,35000.0,Verified,57.96,57000.0,Verified,...,C3,Mar-2018,Current,whole,Cash,21430.15,2324.65,1569.85,754.80,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,owner,10.0,TX,RENT,108000.0,Source Verified,22.28,NaN,NaN,...,A4,Jan-2018,Current,whole,Cash,21586.34,2969.80,2413.66,556.14,0.0
9996,9997,director,8.0,PA,MORTGAGE,121000.0,Verified,32.38,NaN,NaN,...,D3,Feb-2018,Current,whole,Cash,9147.44,1456.31,852.56,603.75,0.0
9997,9998,toolmaker,10.0,CT,MORTGAGE,67000.0,Verified,45.26,107000.0,Source Verified,...,E2,Feb-2018,Current,fractional,Cash,27617.65,4620.80,2382.35,2238.45,0.0
9998,9999,manager,1.0,WI,MORTGAGE,80000.0,Source Verified,11.99,NaN,NaN,...,A1,Feb-2018,Current,whole,Cash,21518.12,2873.31,2481.88,391.43,0.0


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.model_selection import GridSearchCV, train_test_split

features = [
    "loan_amount", "annual_income", "debt_to_income",
    "term", "grade", "verified_income", "homeownership"
]

X = df[features].copy()
y = df["interest_rate"]

for col in X.select_dtypes(include=np.number).columns:
    X[col] = X[col].fillna(X[col].median())
for col in X.select_dtypes(exclude=np.number).columns:
    X[col] = X[col].fillna("missing")

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

ct = ColumnTransformer(
    [
        ("dummify",
         OneHotEncoder(sparse_output=False, handle_unknown='ignore'),
         make_column_selector(dtype_include="object")),
        ("standardize",
         StandardScaler(),
         make_column_selector(dtype_include=np.number))
    ],
    remainder="passthrough"
)

knn_pipeline = Pipeline([
    ("preprocessing", ct),
    ("knn", KNeighborsRegressor())
])

param_grid = {
    "knn__n_neighbors": [5, 10, 15, 20, 25],
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["manhattan", "euclidean"]
}

grid_search = GridSearchCV(
    knn_pipeline,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

best_rmse = np.sqrt(-grid_search.best_score_)
best_params = grid_search.best_params_

print(f"Selected Predictors: {features}")
print(f"Best Parameters Found: {best_params}")
print(f"Cross-Validated Test RMSE: %{best_rmse:.4f}")

Selected Predictors: ['loan_amount', 'annual_income', 'debt_to_income', 'term', 'grade', 'verified_income', 'homeownership']
Best Parameters Found: {'knn__metric': 'manhattan', 'knn__n_neighbors': 5, 'knn__weights': 'distance'}
Cross-Validated Test RMSE: %1.5580
